# Tool-wear compensation: a measured simulation study (toy model)

A cutting tool wears as it makes parts. Can software that measures parts and adjusts the tool offset reduce scrap and lost production, compared with no correction, a fixed tool life, a careful human-style rule, and a Kalman-filter predictor?

**Read this first.** This is a **model with invented numbers**. It says nothing about a real machine, tool or process, and nothing about aerospace or medical precision. Its purpose is to practise honest measurement: settings tuned on one set of seeds, results on different seeds, hard scenarios, paired statistics, and edge-of-grid warnings.

**How to run:** Runtime > Run all. Copy ALL the output text (and `wear-study.png`).

**Section 4** repeats the study on a wear model built on Taylor's tool-life equation (the cutting speed sets the tool life) and tests settings tuned at one speed on two other speeds. Its wear shape and its conversion to part error are assumptions, listed in the code.

## 1. The model, the policies and the safety envelope (max step, max total, minimum gap)

In [ ]:
"""Toy tool-wear compensation study. A MODEL, not a machine: the numbers say nothing about real machines.

One cutting tool makes parts in sequence. Its wear grows with every part (faster as it gets worn). The part
dimension error is  wear - offset + noise.  Every `measure_every` parts one part is measured (with noise and
sometimes a wrong reading). A policy may change the tool offset, subject to a safety envelope enforced outside
the policy (max step, max total offset, minimum parts between changes), and may change the tool (costs downtime).
"""
import math

import numpy as np

TOL = 10.0          # part is scrap if |error| > 10 um
PROC_SIGMA = 1.5    # random scatter of each part, um
TOOL_CHANGE_COST = 15  # parts of lost production per tool change
N_PARTS = 600
ENV = dict(max_step=4.0, max_total=60.0, min_gap=5)  # the "safety envelope": set outside the policy


class ToyWear:
    """The original invented wear model, unchanged: the wear rate grows with the wear, with random scatter and rare
    sudden steps."""
    def __init__(self, base=0.35, accel=0.00035, burst_prob=0.004):
        self.base, self.accel, self.burst_prob = base, accel, burst_prob

    def path(self, rng, n):
        w, out = 0.0, []
        for _ in range(n):
            rate = self.base * (1 + self.accel * w * w) * rng.lognormal(0, 0.25)
            if rng.random() < self.burst_prob:
                rate += rng.uniform(3, 8)  # a chip or a hard spot: sudden wear step
            w += rate
            out.append(w)
        return np.array(out)


TOY_WEAR = ToyWear()


def wear_path(rng, n, base=0.35, accel=0.00035, burst_prob=0.004):  # kept so older code still works
    return ToyWear(base, accel, burst_prob).path(rng, n)


class TaylorWear:
    """A wear hook built on Taylor's tool-life equation  V * T**n = C  (V cutting speed, T tool life in minutes).
    The tool life is T = (C / V) ** (1 / n) and the flank wear VB reaches `vb_limit_mm` at the end of that life
    (ISO 3685 uses a mean VB of 0.3 mm as the end-of-life criterion in tool-life tests).
    ASSUMED, not measured: (1) the shape of the wear curve over the life (quick start, steady middle, accelerating end);
    (2) the part dimension error is the radial recession VB * tan(clearance angle); (3) the cutting time per part.
    The numbers used in this study are illustrative textbook-style values, not any real tool or process."""
    def __init__(self, V, C, n, t_cut_min, vb_limit_mm=0.3, clearance_deg=7.0, sigma=0.25, burst_prob=0.004):
        for name, v in (("V", V), ("C", C), ("n", n), ("t_cut_min", t_cut_min), ("vb_limit_mm", vb_limit_mm)):
            if isinstance(v, bool) or not isinstance(v, (int, float)) or not math.isfinite(v) or v <= 0:
                raise ValueError(f"{name} must be a finite number above zero")
        if isinstance(clearance_deg, bool) or not isinstance(clearance_deg, (int, float)) or not 0 < clearance_deg <= 45:
            raise ValueError("clearance_deg must be above 0 and at most 45")
        if isinstance(sigma, bool) or not isinstance(sigma, (int, float)) or not 0 <= sigma < 1:
            raise ValueError("sigma must be at least 0 and below 1")
        if isinstance(burst_prob, bool) or not isinstance(burst_prob, (int, float)) or not 0 <= burst_prob <= 1:
            raise ValueError("burst_prob must be between 0 and 1")
        self.life_min = (C / V) ** (1.0 / n)
        self.life_parts = self.life_min / t_cut_min
        self.limit_um = vb_limit_mm * math.tan(math.radians(clearance_deg)) * 1000.0
        self.sigma, self.burst_prob = sigma, burst_prob

    @staticmethod
    def shape(u):
        """Fraction of the end-of-life wear reached at the fraction u of the tool life (an assumed shape; shape(1) = 1)."""
        return 0.15 * (1 - np.exp(-u / 0.05)) + 0.70 * u + 0.15 * u ** 4

    def path(self, rng, n):
        mean = self.limit_um * self.shape(np.arange(0, n + 1) / self.life_parts)
        grow = np.diff(mean) * rng.lognormal(0, self.sigma, n)
        burst = (rng.random(n) < self.burst_prob) * rng.uniform(3, 8, n)
        return np.cumsum(grow + burst)


def run(policy, seed, meas_sigma=1.0, outlier_prob=0.0, measure_every=5, n=N_PARTS, trace=False, wear_model=None):
    rng = np.random.default_rng(seed)
    wear_full = (wear_model or TOY_WEAR).path(rng, n + 5 * n)  # enough for tool changes
    noise = rng.normal(0, PROC_SIGMA, n)
    mnoise = rng.normal(0, meas_sigma, n)
    spikes = rng.random(n) < outlier_prob
    spike_val = rng.choice([-1, 1], n) * rng.uniform(15, 30, n)
    offset, tool_start, last_change = 0.0, 0, -10**6
    state = policy.reset()
    scrap = changes = tool_changes = parts_made = 0
    abs_err, signed, offs = [], [], []
    downtime = 0
    k = 0
    while k < n:
        wear = wear_full[k - tool_start + 1] if k - tool_start + 1 < len(wear_full) else wear_full[-1]
        err = wear - offset + noise[k]
        parts_made += 1
        abs_err.append(abs(err)); signed.append(err); offs.append(offset)
        if abs(err) > TOL:
            scrap += 1
        meas = None
        if k % measure_every == 0:
            meas = err + mnoise[k] + (spike_val[k] if spikes[k] else 0.0)
        action = policy.step(state, k - tool_start, meas, offset)
        if action[0] == "offset":
            target = action[1]
            if k - last_change >= ENV["min_gap"]:
                step = float(np.clip(target - offset, -ENV["max_step"], ENV["max_step"]))
                new = float(np.clip(offset + step, -ENV["max_total"], ENV["max_total"]))
                if abs(new - offset) > 1e-9:
                    offset, changes, last_change = new, changes + 1, k
        elif action[0] == "tool":
            tool_changes += 1
            downtime += TOOL_CHANGE_COST
            tool_start, offset, last_change = k + 1, 0.0, k
            state = policy.reset()
        k += 1
    out = dict(scrap=scrap, scrap_rate=scrap / parts_made, mean_abs_err=float(np.mean(abs_err)), offset_changes=changes,
               tool_changes=tool_changes, lost_parts=scrap + downtime)
    if trace:
        out['trace'] = (signed, offs)
    return out


class NoCompensation:
    """Never touches the offset. Changes the tool only when a measurement shows scrap-size error."""
    def reset(self): return {}
    def step(self, s, age, meas, offset):
        return ("tool",) if meas is not None and abs(meas) > TOL else ("none",)


class ThresholdOperator:
    """Like a careful human: when a measurement is outside `frac` of the tolerance, set the offset to cancel it.
    Changes the tool when the offset it would need is almost out of range."""
    def __init__(self, frac=0.6): self.frac = frac
    def reset(self): return {}
    def step(self, s, age, meas, offset):
        if meas is None: return ("none",)
        if offset + meas > 0.9 * ENV["max_total"]: return ("tool",)
        return ("offset", offset + meas) if abs(meas) > self.frac * TOL else ("none",)


class FixedLife:
    """Common shop practice: no offset changes, change the tool after a fixed number of parts."""
    def __init__(self, life=100): self.life = life
    def reset(self): return {}
    def step(self, s, age, meas, offset):
        return ("tool",) if age + 1 >= self.life else ("none",)


class KalmanPredictive:
    """Tracks wear and wear rate with a Kalman filter, ignores readings that disagree wildly (outlier gate), and sets
    the offset ahead of the wear so the error stays centred. Tool change when the offset range is nearly used up."""
    def __init__(self, q_w=0.3, q_r=0.02, r=None, gate=4.0, lead=3, deadband=0.35, meas_sigma=1.0):
        self.q_w, self.q_r, self.gate, self.lead, self.deadband = q_w, q_r, gate, lead, deadband
        self.r = r if r is not None else meas_sigma ** 2 + PROC_SIGMA ** 2
    def reset(self):
        return dict(x=np.array([0.0, 0.3]), P=np.diag([25.0, 1.0]), last=0)
    def step(self, s, age, meas, offset):
        F = np.array([[1.0, 1.0], [0.0, 1.0]])
        dt_steps = max(age - s["last"], 1)
        Fd = np.linalg.matrix_power(F, dt_steps)
        Q = np.diag([self.q_w * dt_steps, self.q_r * dt_steps])
        s["x"], s["P"], s["last"] = Fd @ s["x"], Fd @ s["P"] @ Fd.T + Q, age
        if meas is not None:
            y = (meas + offset) - s["x"][0]  # the measured error plus the offset = the wear seen by the gauge
            S = s["P"][0, 0] + self.r
            if abs(y) <= self.gate * np.sqrt(S):  # outlier gate: wild readings are ignored, not believed
                K = s["P"][:, 0] / S
                s["x"] = s["x"] + K * y
                s["P"] = s["P"] - np.outer(K, s["P"][0])
        wear_ahead = s["x"][0] + self.lead * s["x"][1]
        if wear_ahead > ENV["max_total"] * 0.9: return ("tool",)
        return ("offset", wear_ahead) if abs(wear_ahead - offset) > self.deadband * TOL else ("none",)


def evaluate(make_policy, seeds, **kw):
    rows = [run(make_policy(), s, **kw) for s in seeds]
    return {k: float(np.mean([r[k] for r in rows])) for k in rows[0] if k != 'trace'}, rows


## 1b. Self-checks of the wear models (run before the study; each assertion was broken on purpose once to prove it can fail)

In [ ]:
# Self-checks of the wear models. If any assertion fails, do not trust the study below.
def _reference_toy_path(rng, n, base=0.35, accel=0.00035, burst_prob=0.004):
    """The original loop, copied here unchanged, so the toy model can be compared with it on any numpy version."""
    w, out = 0.0, []
    for _ in range(n):
        rate = base * (1 + accel * w * w) * rng.lognormal(0, 0.25)
        if rng.random() < burst_prob:
            rate += rng.uniform(3, 8)
        w += rate
        out.append(w)
    return np.array(out)


class _NoWear:
    def path(self, rng, n):
        return np.zeros(n)


def self_checks():
    n_checks = 0
    # 1. the toy model is exactly the original one
    for s in range(5):
        assert np.array_equal(ToyWear().path(np.random.default_rng(s), 80), _reference_toy_path(np.random.default_rng(s), 80)), "toy wear changed"
    assert np.array_equal(wear_path(np.random.default_rng(3), 40), _reference_toy_path(np.random.default_rng(3), 40)), "wear_path changed"
    assert np.array_equal(wear_path(np.random.default_rng(3), 40, base=0.5, accel=0.001, burst_prob=0.05),
                          _reference_toy_path(np.random.default_rng(3), 40, base=0.5, accel=0.001, burst_prob=0.05)), "wear_path ignores its arguments"
    n_checks += 1
    # 2. run() uses the model it is given, and without one it uses the toy model
    assert run(NoCompensation(), 7) == run(NoCompensation(), 7, wear_model=ToyWear()), "default is not the toy model"
    assert run(NoCompensation(), 7, wear_model=_NoWear())["scrap"] == 0 < run(NoCompensation(), 7)["scrap"], "run ignores wear_model"
    n_checks += 1
    # 3. Taylor tool life: V * T**n = C, and doubling the speed divides the life by 2**(1/n)
    m = TaylorWear(V=250, C=500, n=0.25, t_cut_min=0.1)
    assert abs(m.life_min - 16.0) < 1e-9 and abs(m.life_parts - 160.0) < 1e-6, "tool life formula wrong"
    fast = TaylorWear(V=500, C=500, n=0.25, t_cut_min=0.1)
    assert abs(fast.life_min / m.life_min - 2 ** (-1 / 0.25)) < 1e-12, "speed law wrong"
    assert abs(m.limit_um - 0.3 * math.tan(math.radians(7.0)) * 1000.0) < 1e-9, "limit in um wrong"
    n_checks += 1
    # 4. the assumed wear shape: reaches 1 at the end of the life, rises all the way
    assert abs(TaylorWear.shape(1.0) - 1.0) < 1e-8 and TaylorWear.shape(0.0) == 0.0, "shape end points wrong"
    assert np.all(np.diff(TaylorWear.shape(np.linspace(0, 1, 1001))) > 0), "shape not rising"
    # the interior of the curve is an ASSUMPTION; this pins it so nobody changes it by accident
    assert abs(TaylorWear.shape(0.5) - (0.15 * (1 - math.exp(-10.0)) + 0.35 + 0.15 * 0.0625)) < 1e-12, "assumed shape changed"
    assert abs(TaylorWear.shape(0.1) - (0.15 * (1 - math.exp(-2.0)) + 0.07 + 0.15 * 0.0001)) < 1e-12, "assumed shape changed"
    n_checks += 1
    # 5. without noise the wear reaches the criterion exactly at the end of the life (160 parts), not before
    q = TaylorWear(V=250, C=500, n=0.25, t_cut_min=0.1, sigma=0.0, burst_prob=0.0).path(np.random.default_rng(0), 200)
    assert abs(q[159] - m.limit_um) < 1e-6 * m.limit_um and q[158] < m.limit_um < q[160], "wear does not reach the criterion at the end of life"
    n_checks += 1
    # 6. with noise the wear never goes down, and sudden steps really happen
    noisy = TaylorWear(V=250, C=500, n=0.25, t_cut_min=0.1).path(np.random.default_rng(1), 500)
    assert noisy[0] >= 0 and np.all(np.diff(noisy) >= 0), "wear went down"
    steps = TaylorWear(V=250, C=500, n=0.25, t_cut_min=0.1, burst_prob=1.0).path(np.random.default_rng(1), 50)
    assert np.all(np.diff(steps) >= 3.0), "sudden steps missing"
    n_checks += 1
    # 7. same seed, same path; other seed, other path
    a, b, c = (TaylorWear(V=250, C=500, n=0.25, t_cut_min=0.1).path(np.random.default_rng(s), 60) for s in (4, 4, 5))
    assert np.array_equal(a, b) and not np.array_equal(a, c), "not reproducible by seed"
    n_checks += 1
    # 8. bad parameters are refused
    good = dict(V=250, C=500, n=0.25, t_cut_min=0.1)
    for bad in ({"V": 0}, {"V": -1}, {"V": float("nan")}, {"V": float("inf")}, {"V": True}, {"V": "250"}, {"C": 0}, {"n": 0}, {"n": -0.2},
                {"t_cut_min": 0}, {"vb_limit_mm": 0}, {"clearance_deg": 0}, {"clearance_deg": 46}, {"clearance_deg": float("nan")},
                {"clearance_deg": True}, {"clearance_deg": "7"}, {"sigma": -0.1}, {"sigma": 1.0}, {"sigma": False}, {"sigma": "0.2"},
                {"burst_prob": -0.1}, {"burst_prob": 1.1}, {"burst_prob": True}, {"burst_prob": False}, {"burst_prob": "0.1"}):
        try:
            TaylorWear(**{**good, **bad})
        except ValueError:
            continue
        raise AssertionError(f"accepted bad parameters {bad}")
    n_checks += 1
    return n_checks


print(f"all {self_checks()} groups of self-checks passed")


## 2. Tuning, held-out results, hard scenarios, trade-off, paired comparison

In [ ]:
# Study cells (run after wear_sim.py): tuning, held-out evaluation, hard scenarios, trade-off, paired comparison.
import numpy as np
TUNE, TEST = range(0, 50), range(1000, 1200)

def best(grid, mk, key="lost_parts"):
    res = []
    for p in grid:
        m, _ = evaluate(lambda: mk(**p), TUNE)
        res.append((m[key], p))
    return min(res, key=lambda t: t[0])

def edge_warning(name, chosen, grids):
    for k, values in grids.items():
        if chosen[k] in (min(values), max(values)):
            print(f"  WARNING: {name} {k}={chosen[k]} is at the edge of its search grid {values}: the true optimum may lie outside.")

G_OP = {"frac": (0.05, 0.1, 0.2, 0.3, 0.4, 0.5, 0.6)}
G_FL = {"life": (15, 20, 30, 40, 60, 80, 100, 150, 250)}
G_KF = {"lead": (0, 1, 2, 3, 5), "deadband": (0.05, 0.1, 0.2, 0.35, 0.5), "q_w": (0.03, 0.1, 0.3, 1.0)}
op = best([dict(frac=f) for f in G_OP["frac"]], ThresholdOperator)
fl = best([dict(life=L) for L in G_FL["life"]], FixedLife)
kf = best([dict(lead=l, deadband=d, q_w=q) for l in G_KF["lead"] for d in G_KF["deadband"] for q in G_KF["q_w"]], KalmanPredictive)
print("Settings tuned on seeds 0-49, base scenario:")
print("  operator rule :", op[1], "| fixed tool life:", fl[1], "| kalman:", kf[1])
edge_warning("operator", op[1], G_OP); edge_warning("fixed life", fl[1], G_FL); edge_warning("kalman", kf[1], G_KF)
POL = {"no compensation": NoCompensation, "fixed tool life": lambda: FixedLife(**fl[1]),
       "operator rule": lambda: ThresholdOperator(**op[1]), "kalman predictive": lambda: KalmanPredictive(**kf[1])}

def table(title, **kw):
    print("\n" + title + "  (held-out seeds 1000-1199, mean per 600 parts)")
    out = {}
    for n, mk in POL.items():
        m, rows = evaluate(mk, TEST, **kw)
        out[n] = rows
        print(f"  {n:18s} lost parts={m['lost_parts']:6.1f}  scrap={m['scrap']:6.1f}  offset changes={m['offset_changes']:5.1f}  tool changes={m['tool_changes']:5.1f}")
    return out

base = table("BASE: gauge noise 1 um, measure every 5 parts")
table("Noisier gauge: 3 um", meas_sigma=3.0)
table("Very noisy gauge: 6 um", meas_sigma=6.0)
table("5% wrong readings (spikes of 15-30 um)", outlier_prob=0.05)
table("Sparse measurement: every 25 parts", measure_every=25)

print("\nTRADE-OFF: fewer offset changes versus more scrap (operator rule and kalman, base scenario)")
for f in (0.2, 0.4, 0.6, 0.8):
    m, _ = evaluate(lambda: ThresholdOperator(frac=f), TEST)
    print(f"  operator frac={f:.1f}: changes={m['offset_changes']:5.1f}  scrap={m['scrap']:6.1f}  lost parts={m['lost_parts']:6.1f}")
for d in (0.2, 0.5, 1.0, 1.5):
    m, _ = evaluate(lambda: KalmanPredictive(**{**kf[1], "deadband": d}), TEST)
    print(f"  kalman deadband={d:.1f}: changes={m['offset_changes']:5.1f}  scrap={m['scrap']:6.1f}  lost parts={m['lost_parts']:6.1f}")

def paired(a, b, key="lost_parts", n_boot=2000):
    d = np.array([x[key] - y[key] for x, y in zip(base[a], base[b])]); rng = np.random.default_rng(0)
    boots = [rng.choice(d, len(d)).mean() for _ in range(n_boot)]
    lo, hi = np.percentile(boots, [2.5, 97.5])
    print(f"  {a} minus {b}: mean difference {d.mean():+.1f} lost parts, 95% interval [{lo:+.1f}, {hi:+.1f}] over {len(d)} seeds")
print("\nPAIRED COMPARISON on the same seeds (base scenario). If the interval includes 0, there is no clear difference:")
paired("kalman predictive", "operator rule")
paired("operator rule", "no compensation")
paired("kalman predictive", "no compensation")
print("\nThis is a toy model with invented numbers (tolerance 10 um, tool change = 15 lost parts, wear model made up).")
print("It says nothing about a real machine, a real tool or a real process.")


## 4. A published wear model as a hook: Taylor's tool-life equation, speed changes after tuning

In [ ]:
# Section 4: the same four policies on a Taylor-style wear model, where the cutting speed sets the tool life.
# ILLUSTRATIVE numbers: C = 500 m/min and n = 0.25 are textbook-style values for carbide on steel, t_cut = 0.1 min per part is a choice.
TAYLOR = dict(C=500.0, n=0.25, t_cut_min=0.1)
SPEEDS = (200.0, 250.0, 300.0)   # m/min; settings are tuned at 250 and then tested at all three
print("Taylor-style wear model (assumed shape; VB limit 0.3 mm as in ISO 3685; radial wear = VB * tan(7 degrees)):")
for V in SPEEDS:
    t = TaylorWear(V=V, **TAYLOR)
    print(f"  V={V:.0f} m/min: tool life {t.life_min:5.1f} min = {t.life_parts:5.0f} parts, wear at end of life {t.limit_um:4.1f} um (tolerance +/-{TOL:.0f} um)")

def best_for(grid, mk, model, key="lost_parts"):
    res = []
    for p in grid:
        m, _ = evaluate(lambda: mk(**p), TUNE, wear_model=model)
        res.append((m[key], p))
    return min(res, key=lambda t: t[0])

tune_model = TaylorWear(V=250.0, **TAYLOR)
op_t = best_for([dict(frac=f) for f in G_OP["frac"]], ThresholdOperator, tune_model)
fl_t = best_for([dict(life=L) for L in G_FL["life"]], FixedLife, tune_model)
kf_t = best_for([dict(lead=l, deadband=d, q_w=q) for l in G_KF["lead"] for d in G_KF["deadband"] for q in G_KF["q_w"]], KalmanPredictive, tune_model)
print("\nSettings tuned on seeds 0-49 at V=250 m/min:")
print("  operator rule :", op_t[1], "| fixed tool life:", fl_t[1], "| kalman:", kf_t[1])
edge_warning("operator", op_t[1], G_OP); edge_warning("fixed life", fl_t[1], G_FL); edge_warning("kalman", kf_t[1], G_KF)
POL_T = {"no compensation": NoCompensation, "fixed tool life": lambda: FixedLife(**fl_t[1]),
         "operator rule": lambda: ThresholdOperator(**op_t[1]), "kalman predictive": lambda: KalmanPredictive(**kf_t[1])}
for V in SPEEDS:
    model = TaylorWear(V=V, **TAYLOR)
    print(f"\nV={V:.0f} m/min (tool life {model.life_parts:.0f} parts)  held-out seeds 1000-1199, mean per 600 parts" + ("  [tuned here]" if V == 250.0 else "  [speed changed after tuning]"))
    results, rows_by = {}, {}
    for name, mk in POL_T.items():
        m, rows_by[name] = evaluate(mk, TEST, wear_model=model)
        results[name] = m
        print(f"  {name:18s} lost parts={m['lost_parts']:6.1f}  scrap={m['scrap']:6.1f}  offset changes={m['offset_changes']:5.1f}  tool changes={m['tool_changes']:5.1f}")
    ranking = sorted(results, key=lambda k: results[k]["lost_parts"])
    print("  ranking by lost parts (best first):", " < ".join(ranking), "(small gaps may be noise: see the paired interval)")
    d = np.array([x["lost_parts"] - y["lost_parts"] for x, y in zip(rows_by["kalman predictive"], rows_by["operator rule"])])
    rng_b = np.random.default_rng(0)
    boots = [rng_b.choice(d, len(d)).mean() for _ in range(2000)]
    lo, hi = np.percentile(boots, [2.5, 97.5])
    print(f"  paired: kalman predictive minus operator rule: mean {d.mean():+.1f} lost parts, 95% interval [{lo:+.1f}, {hi:+.1f}] over {len(d)} seeds" + ("" if lo > 0 or hi < 0 else "  (includes 0: no clear difference)"))
print("\nThe wear model is an assumption built on a published tool-life equation, with an assumed wear shape and an assumed conversion to part error.")
print("It is still a toy: it says nothing about a real machine, a real tool or a real process.")


## 3. What the error looks like

In [ ]:
# Error of each part over time for one seed: how each policy keeps the dimension inside the tolerance band.
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
fig, axes = plt.subplots(3, 1, figsize=(9, 7), sharex=True)
for ax, (name, mk) in zip(axes, [("no compensation", NoCompensation), ("operator rule", POL["operator rule"]), ("kalman predictive", POL["kalman predictive"])]):
    r = run(mk(), 1000, trace=True)
    err, offs = r["trace"]
    ax.plot(err, lw=0.8, label="part error (um)")
    ax.axhline(TOL, color="r", ls="--", lw=0.8); ax.axhline(-TOL, color="r", ls="--", lw=0.8)
    ax.set_ylabel(name, fontsize=8); ax.set_ylim(-35, 35)
    ax.set_title(f"scrap {r['scrap']}, offset changes {r['offset_changes']}, tool changes {r['tool_changes']}", fontsize=8, loc="right")
axes[-1].set_xlabel("part number (red lines = tolerance; this is a toy model with invented numbers)")
fig.tight_layout(); fig.savefig("wear-study.png", dpi=110)
try:
    from IPython.display import Image, display
    display(Image("wear-study.png"))
except ImportError:
    print("plot saved as wear-study.png")
